[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/01_%E5%A3%B2%E4%B8%8A%E3%83%87%E3%83%BC%E3%82%BF%E5%88%86%E6%9E%90.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 実践マーケ-01. BtoB売上データ分析ワークシート

**舞台設定**：あなたは法人向けソフトを売る会社のデータ分析担当。
半年分の商談データ `sales_btob.csv` を分析し、社内会議で報告します。

**使う統計（4〜3級）**：合計・平均・構成比・クロス集計・グループ別比較・相関

> BtoB（Business to Business）= 会社が会社に売るビジネス。1件の金額が大きく、受注までの検討が長いのが特徴です。

In [ ]:
import pandas as pd              # 表データ
import numpy as np               # 数値計算
import matplotlib.pyplot as plt  # グラフ描画
plt.rcParams['axes.unicode_minus'] = False       # マイナス記号の文字化けを防ぐ
btob = pd.read_csv('../data/sales_btob.csv')      # 商談データを読み込む
print('商談数:', len(btob))      # 行数（商談件数）
btob.head()                      # 先頭5行を確認

### 使うデータ：`sales_btob.csv`（架空のBtoB商談400件）
1行＝商談1件。`受注`は 1=成約 / 0=失注。

| 商談ID | 受注日 | 業界 | 獲得チャネル | 商談金額 | 担当者 | 受注 |
|---|---|---|---|---|---|---|
| D0001 | 2026-01-15 | 小売 | 紹介 | 1,211,000 | 佐藤 | 0 |
| D0002 | 2026-02-05 | 医療 | テレアポ | 400,000 | 田中 | 0 |
| D0003 | 2026-02-19 | IT | 紹介 | 542,000 | 高橋 | 1 |

## 1. まずデータを知る

分析の第一歩は「どんな列があり、どんな値か」を把握することです。

In [ ]:
btob.info()                                # 各列の型・欠損の有無
btob.describe(include='all')               # 数値・文字すべての要約

## 2. 全体のKPIを出す

KPI = 重要な指標。BtoB営業では次がよく使われます。
- 受注率（受注した商談の割合）
- 受注金額の合計（売上）
- 平均商談単価

In [ ]:
won = btob[btob['受注'] == 1]              # 受注（成約）した商談だけ抽出
print('受注率: {:.1%}'.format(btob['受注'].mean()))   # 受注=1の割合
print('受注件数:', len(won))                # 成約件数
print('売上合計: {:,} 円'.format(won['商談金額'].sum()))   # 受注金額の合計
print('平均単価: {:,.0f} 円'.format(won['商談金額'].mean()))  # 1件あたり平均

## 3. チャネル別に分解する（どこから売上が来たか）

「獲得チャネル」ごとに受注売上を集計し、棒グラフにします。

In [ ]:
by_ch = won.groupby('獲得チャネル')['商談金額'].sum().sort_values(ascending=False)  # チャネル別の売上（多い順）
print(by_ch)
by_ch.plot(kind='bar', figsize=(7,4), title='チャネル別の受注売上')  # 棒グラフ
plt.ylabel('売上(円)'); plt.xticks(rotation=0); plt.show()

### 構成比（パレート分析）
「売上の8割はどのチャネルから？」を割合で見ます。

In [ ]:
ratio = (by_ch / by_ch.sum() * 100).round(1)  # 各チャネルの売上構成比(%)
cum = ratio.cumsum()                          # 上から積み上げた累積(%)
pd.DataFrame({'売上構成比%': ratio, '累積%': cum})  # パレート分析の表

## 4. 月ごとの推移を見る（時系列）

受注日を月単位にまとめ、売上の伸びを折れ線で確認します。

In [ ]:
won = won.copy()                           # コピーして警告を防ぐ
won['受注月'] = pd.to_datetime(won['受注日']).dt.to_period('M').astype(str)  # 受注日を「年月」に変換
monthly = won.groupby('受注月')['商談金額'].sum()  # 月ごとの売上合計
monthly.plot(marker='o', figsize=(7,4), title='月別売上の推移')  # 折れ線グラフ
plt.ylabel('売上(円)'); plt.grid(alpha=0.3); plt.show()
monthly

## 5. 受注件数と売上の関係を相関で見る

「受注件数が多い月ほど売上も大きい」と言えるでしょうか。**相関係数**で確かめます。相関係数は −1〜+1 で、+1に近いほど右肩上がりの関係が強いことを表します（統計3級）。

In [ ]:
受注件数 = won.groupby('受注月').size()                    # 月ごとの受注件数
月次 = pd.DataFrame({'受注件数': 受注件数, '売上': monthly})  # 受注件数と売上を月単位で並べる
r = 月次['受注件数'].corr(月次['売上'])
print(f'受注件数と売上の相関係数 r = {r:.3f}')
月次.plot(x='受注件数', y='売上', kind='scatter', figsize=(6,4), title='月別 受注件数 vs 売上'); plt.show()
print('（目安）|r|>0.7 で強い・0.4〜0.7でやや強い・<0.2でほぼ無関係')

---
## ワークシート課題

**課題1.** 業界別の受注売上を集計し、最も売れている業界トップ3を答えよう。

**課題2.** 担当者別の受注件数と受注率を出し、「最も成績が良い担当者」を見つけよう。

**課題3.**（報告書）この会社の社長に向けて、分かったことを3行でまとめよう。
「①売上の柱は◯◯チャネル ②◯◯業界が有望 ③△△を強化すべき」のように。

In [ ]:
# 課題1


In [ ]:
# 課題2


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[01_売上データ分析 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/05_%E5%AE%9F%E8%B7%B5_BtoB%E3%83%9E%E3%83%BC%E3%82%B1/01_%E5%A3%B2%E4%B8%8A%E3%83%87%E3%83%BC%E3%82%BF%E5%88%86%E6%9E%90.md)**

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""      # ← あなたの名前（例: 山田）
LOG_URL = ""   # ← 記録用URL（配布時に設定。空なら記録せず表示のみ）
NOTEBOOK = "05_実践_BtoBマーケ/01_売上データ分析"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")